# 03e — Validación: estabilidad (§7A) + utilidad (§7B)

¿Existe la tipología y serviría para predecir? Solo se acepta si pasa ambas (7A ∧ 7B). Candidatos: conjunto A con k en {5, 7, 12} — el único que supera al control CCAA en severidad con NMI bajo (B es el calendario con pasos extra, C casi nunca supera los controles; ver 03b). Salida: `unsupervised/validacion_resumen.csv` + veredicto explícito (aceptada / rechazada / no concluyente).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.typology import (
    NMI_TRIVIALITY_MAX, PREPROCESSING_ARI_MIN, STABILITY_ARI_MIN,
    anchor_min_delta, cross_year_profiles, fit_labels, gmm_overlap_mass,
    leave_one_group_out, noncoincidence_ari, nontriviality_nmi,
    pairwise_ari, resample_stability, severity_separation, validation_verdict,
)
from wildfire.processing.unsupervised import ID_COL, check_ids, fit_gmm_range, negative_control_partitions
from wildfire.processing.typology import SPATIAL_CORR_MIN
from wildfire.processing.unsupervised import load_min_cliff_delta


## 03e.1 — Candidatos y etiquetas base

Se carga el conjunto A (PCs) y se estandariza una vez por experimento (cada escalado se ajusta aquí, nunca se reutiliza el de `normalized.csv`). Las etiquetas al k ilustrativo 5 vienen de `cluster_labels.csv`; k=7 y k=12 se ajustan aquí mismo con la misma semilla.


In [ ]:
SHORTLIST = [5, 7, 12]
feat_a = load_csv("data/processed/unsupervised/features_a.csv")
perfil = load_csv("data/processed/unsupervised/perfilado.csv")
cl = load_csv("data/processed/unsupervised/cluster_labels.csv")
check_ids(feat_a, perfil, cl)
assert len(feat_a) == len(perfil) == len(cl)
Xa = feat_a.drop(columns=[ID_COL]).to_numpy(dtype=float)
Xs = StandardScaler().fit_transform(Xa)
etiquetas = {"k5": cl["kmeans_a_k5"].to_numpy()}
for k in [7, 12]:
    etiquetas[f"k{k}"] = fit_labels(Xs, k)
frp = perfil["frp"].to_numpy()
print(f"filas: {len(feat_a)} | variantes: {sorted(etiquetas)}")


## 03e.2 — 7A Estabilidad por remuestreo

Cómo leerlo: 30 reajustes al 80 %, ARI contra las etiquetas con datos completos; la barra es media > 0,7. Un ARI alto dice que la estructura está en los datos, no en la muestra.


In [ ]:
filas7a, filas7b = [], []
for nombre, lab in etiquetas.items():
    r = resample_stability(Xa, int(nombre[1:]), n_resamples=30, frac=0.8)
    ok = r["mean_ari"] > STABILITY_ARI_MIN
    filas7a.append({"check": "resample", "variante": nombre, "valor": round(r["mean_ari"], 3),
                    "min": round(r["min_ari"], 3), "pasa": ok})
    print(f"{nombre}: media={r['mean_ari']:.3f} min={r['min_ari']:.3f} -> {'pasa' if ok else 'falla'}")


## 03e.3 — 7A Estabilidad espacial (una CCAA fuera cada vez)

Se reajusta sin cada CCAA: tienen que sobrevivir los k grupos con los mismos perfiles (correlación con emparejamiento húngaro, que ignora el reetiquetado). Si algún grupo desaparece, es un artefacto regional, no una tipología.


In [ ]:
for nombre, lab in etiquetas.items():
    t = leave_one_group_out(Xa, perfil["ccaa"], int(nombre[1:]))
    ok = bool((t["groups_found"] == int(nombre[1:])).all()
            and (t["profile_corr"] > SPATIAL_CORR_MIN).all())
    filas7a.append({"check": "espacial", "variante": nombre,
                    "valor": round(float(t["profile_corr"].min()), 3),
                    "min": round(float(t["profile_corr"].min()), 3), "pasa": ok})
    peor = t.sort_values("profile_corr").iloc[0]
    print(f"{nombre}: grupos siempre {int(nombre[1:])}/{int(nombre[1:])} = "
          f"{bool((t['groups_found'] == int(nombre[1:])).all())}, peor corr {peor['group']} "
          f"({peor['profile_corr']:.3f}) -> {'pasa' if ok else 'falla'}")


## 03e.4 — 7A Réplica temporal (2023 ↔ 2024)

Se ajusta en un año y se asigna el otro, en ambas direcciones: si los perfiles por grupo coinciden (correlación alta entre las dos tablas de medias), los regímenes no son de un solo año.


In [ ]:
for nombre, lab in etiquetas.items():
    out = cross_year_profiles(Xa, perfil["year"], int(nombre[1:]))
    ok = out["profile_corr"] > 0.5  # listón de juicio (el plan no fija número: replicación moderada)
    filas7a.append({"check": "temporal", "variante": nombre,
                    "valor": round(out["profile_corr"], 3), "min": None, "pasa": ok})
    print(f"{nombre}: corr entre perfiles {out['profile_corr']:.3f} -> {'pasa' if ok else 'falla'}")


## 03e.5 — 7A Robustez de preprocesado

Standard contra Robust en cada variante (matriz de ARI, barra 0,6 en cada par). La red a nivel crudo (±BLH, ±log1p) colapsó con la migración a PCs: esas decisiones viven ahora dentro del PCA ajustado, idéntico para todos los consumidores, así que la red relevante es la de escalado.


In [ ]:
for nombre, lab in etiquetas.items():
    Xr = RobustScaler().fit_transform(Xa)
    variantes = {"standard": lab, "robust": fit_labels(Xr, int(nombre[1:]))}
    m = pairwise_ari(variantes)
    minimo = float(m.where(~np.eye(len(m), dtype=bool)).min().min())
    ok = minimo > PREPROCESSING_ARI_MIN
    filas7a.append({"check": "preprocesado", "variante": nombre,
                    "valor": round(minimo, 3), "min": None, "pasa": ok})
    print(f"{nombre}: ARI mínimo entre variantes {minimo:.3f} -> {'pasa' if ok else 'falla'}")


## 03e.6 — 7A Sentido físico (revisión con proxy automático)

Medias por grupo en unidades originales (del índice, no de los PCs): el proxy exige lo que la física exige — a más VPD, menos humedad y más ET0 entre grupos (solo esas tres variables entran en la puerta; viento y precipitación se miran en la tabla pero no puntúan). Si el proxy falla, la revisión manual manda (ver tabla).


In [ ]:
crudas = ["temperature_2m", "relative_humidity_2m", "vapour_pressure_deficit",
          "et0_fao_evapotranspiration", "wind_speed_10m", "precipitation"]
lag = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
check_ids(lag, feat_a)
assert len(lag) == len(feat_a)
fisico = {}
for nombre, lab in etiquetas.items():
    medias = lag[crudas].assign(grupo=lab).groupby("grupo").mean()
    display(medias.round(3))
    ok = bool((np.corrcoef(medias["vapour_pressure_deficit"], medias["relative_humidity_2m"])[0, 1] < 0)
              and (np.corrcoef(medias["vapour_pressure_deficit"], medias["et0_fao_evapotranspiration"])[0, 1] > 0))
    fisico[nombre] = ok
    filas7a.append({"check": "fisico", "variante": nombre, "valor": None, "min": None, "pasa": ok})
    print(f"{nombre}: proxy físico -> {'pasa' if ok else 'falla'}")


## 03e.7 — 7B Separación de severidad y controles

Kruskal sobre FRP + máximo |δ| de Cliff por variante, contra las mismas cuentas en los controles triviales (mes y CCAA). La candidata tiene que separarse y superar a ambos; si no, redescubrió el calendario o el mapa.


In [ ]:
controles = negative_control_partitions(perfil)
sev_controles = {nombre: severity_separation(part, frp) for nombre, part in controles.items()}
for nombre, v in sev_controles.items():
    print(f"control {nombre}: max|d|={v['max_abs_delta']:.3f} (par {v['pair']})")
sev_cand = {}
for nombre, lab in etiquetas.items():
    v = severity_separation(lab, frp)
    sev_cand[nombre] = v
    ok = v["passes"] and all(v["max_abs_delta"] > c["max_abs_delta"] for c in sev_controles.values())
    filas7b.append({"check": "severidad", "variante": nombre,
                    "valor": round(v["max_abs_delta"], 3), "min": None, "pasa": ok})
    print(f"{nombre}: max|d|={v['max_abs_delta']:.3f} (par {v['pair']}) -> {'pasa' if ok else 'falla'}")


## 03e.8 — 7B No-coincidencia, no-trivialidad y solape GMM

Tres formas de pillar a un impostor: ARI alto con el calendario/mapa (es el calendario), NMI > 0,5 con una sola variable (es esa variable), y masa de probabilidad GMM difusa (los regímenes se solapan tanto que la etiqueta dura miente). Para k=7 y k=12 la GMM se reajusta aquí (el fichero solo trae k=5).


In [ ]:
part_mes = perfil["month"].to_numpy()
part_ccaa = perfil["ccaa"].to_numpy()
probas = {"k5": cl[[c for c in cl.columns if c.startswith("gmm_a_k5_full_p")]].to_numpy()}
for k in [7, 12]:
    r = fit_gmm_range(pd.DataFrame(Xs), [k], covariance_types=("full",))
    probas[f"k{k}"] = r[(k, "full")]["proba"]
for nombre, lab in etiquetas.items():
    nc = noncoincidence_ari(lab, part_mes, part_ccaa)
    ok_nc = max(nc.values()) < 0.5
    filas7b.append({"check": "no_coincidencia", "variante": nombre,
                    "valor": round(max(nc.values()), 3), "min": None, "pasa": ok_nc})
    nt = nontriviality_nmi(lab, ccaa=part_ccaa, month=part_mes, year=perfil["year"].to_numpy(),
                           daynight=perfil["daynight"].to_numpy())
    filas7b.append({"check": "no_trivialidad", "variante": nombre,
                    "valor": round(max(v for k_, v in nt.items() if k_ != "passes"), 3),
                    "min": None, "pasa": nt["passes"]})
    ov = gmm_overlap_mass(probas[nombre])
    filas7b.append({"check": "solape_gmm", "variante": nombre,
                    "valor": round(ov["frac_below"], 3), "min": None, "pasa": ov["passes"]})
    print(f"{nombre}: ARI vs controles {nc} | NMI max "
          f"{max(v for k_, v in nt.items() if k_ != 'passes'):.3f} | masa difusa {ov['frac_below']:.3f}")


## 03e.9 — Anclaje del listón, tabla resumen y veredicto

El listón de |δ| se ancla justo por encima del mejor control (redondeo a pasos de 0,05 para no perseguir decimales). La tabla `validacion_resumen.csv` deja cada comprobación × variante con su pasa/falla, y el veredicto exige 7A ∧ 7B: sin 7A se rechaza, con 7A pero sin 7B queda no concluyente. Entre aceptadas gana el k menor (interpretabilidad). Nota: el pase de k5 en severidad es un filo (0,714 contra listón 0,70) — irrelevante para el veredicto porque k5 cae en 7A, pero avisa de no sobreleerlo.


In [ ]:
mejor = max(sev_cand.items(), key=lambda kv: kv[1]["max_abs_delta"])
ancla = anchor_min_delta(mejor[1]["max_abs_delta"],
                         {n: v["max_abs_delta"] for n, v in sev_controles.items()})
print(f"ancla: mejor control {ancla['best_control']}={ancla['best_control_max']:.3f} -> "
      f"listón recomendado {ancla['recommended_bar']:.2f} (config actual: {load_min_cliff_delta()})")
print(f"filo k5: max|d|={sev_cand['k5']['max_abs_delta']:.3f} contra listón "
      f"{load_min_cliff_delta()} (margen {(sev_cand['k5']['max_abs_delta'] - load_min_cliff_delta()):.3f}): "
      "su pase en severidad es frágil, pero el veredicto no depende de él (k5 cae en 7A)")

resumen = pd.DataFrame(
    [{"bloque": "7A", **f} for f in filas7a] + [{"bloque": "7B", **f} for f in filas7b]
)
save_csv(resumen, "data/processed/unsupervised/validacion_resumen.csv")
display(resumen)

veredictos = {}
for nombre in etiquetas:
    a = {f["check"]: f["pasa"] for f in filas7a if f["variante"] == nombre}
    b = {f["check"]: f["pasa"] for f in filas7b if f["variante"] == nombre}
    veredictos[nombre] = validation_verdict(a, b)
print(veredictos)
aceptadas = sorted([n for n, v in veredictos.items() if v == "accepted"],
                   key=lambda n: int(n[1:]))
ELEGIDA = aceptadas[0] if aceptadas else None
print(f"ELEGIDA: {ELEGIDA}")
if ELEGIDA is not None:
    lab = etiquetas[ELEGIDA]
    perfiles_fin = lag[crudas].assign(grupo=lab).groupby("grupo").mean().round(3)
    save_csv(perfiles_fin, "data/processed/unsupervised/perfiles_tipologia.csv")
    display(perfiles_fin)


## Cierre 03e

Si hay tipología elegida, `cluster_meteo` y `score_anomalia` pasan a candidatas del modelo de ocurrencia — pero la promoción final exige el contrato de ablación §7B (ΔPR-AUC con/sin esas columnas bajo `blocked_splits`, cuando existan los días de fondo). Hasta entonces, existen pero se declaran no probadas. Alcance: §7 cubre solo la tipología; las puntuaciones de anomalía y de evento no pasan por ninguna puerta aquí y entrarán, si acaso, por el contrato de ablación.
